In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('UserBehavior.csv', header=None)
df.columns = ['user_id', 'item_id', 'category_id','behavior_type', 'timestamp']
df['datetime'] = pd.to_datetime(df['timestamp'], unit='s',errors='coerce')
df = df[(df['datetime'] >= 'xxxx-xx-xx') & (df['datetime'] <= 'xxxx-xx-xx')]
df['date'] = df['datetime'].dt.date

print(df.head())
print(df.info())
print(df.tail())
print(df.isnull().sum())
print(df['date'].min())
print(df['date'].max())
print(df['date'].describe())

# 日单位活跃图
daily_total_actions = df.groupby('date').size()
print(daily_total_actions)
print(daily_total_actions.max())

plt.figure(figsize=(15, 6))
daily_total_actions.plot(marker='o', color='teal')
plt.title('Total Daily User Behavior')
plt.xlabel('Date')
plt.ylabel(' Number of User Behavior')
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 新增用户统计
buy_df = df[df['behavior_type'] == 'buy']
first_buy = buy_df.groupby('user_id')['date'].min()
new_users_per_day = first_buy.value_counts().sort_index()

plt.figure(figsize=(10, 4))
new_users_per_day.plot(kind='bar')
plt.title('Acquisition')
plt.xlabel('Date')
plt.ylabel('number of New User')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# bar漏斗图(原始数据）
funnel_raw = df['behavior_type'].value_counts()
plt.figure(figsize=(8, 5))
funnel_raw = funnel_raw[['pv', 'fav', 'cart', 'buy']]/1e7
funnel_raw.plot(kind='barh', color='skyblue')
plt.title('User Behavior Funnel (raw)')
plt.xlabel('Number of Behaviors(100k)')
plt.ylabel('Type of Behavior')
plt.tight_layout()
plt.show()

# bar漏斗图(去重数据）
funnel_unique = df.groupby(['user_id', 'behavior_type']).size().reset_index()
funnel_unique_counts = funnel_unique['behavior_type'].value_counts()
funnel_unique_counts = funnel_unique_counts[['pv', 'fav', 'cart', 'buy']]/1e7
plt.figure(figsize=(8, 5))
funnel_unique_counts.plot(kind='barh', color='skyblue')
plt.title('User Behavior Funnel (de-duplicate)')
plt.xlabel('Number of Behaviors(100k)')
plt.ylabel('Type of Behavior')
plt.tight_layout()
plt.show()

pv_df = df[df['behavior_type'] == 'pv']
pv_counts = pv_df['user_id'].value_counts()
one_click_users = (pv_counts == 1).sum()
total_pv_users =pv_df['user_id'].nunique()
print(pv_counts)
print(one_click_users)
bounce_rate = one_click_users / total_pv_users
bounce_rate_percent = bounce_rate * 100
print(f"{bounce_rate_percent:.5f}%")